# Summary Evaluation Benchmark

|Metric | Scope Description |
|-----|------|
|BERTScore | Semantic similarity between LLM summaries and human-written summaries |
|Hallucination Rate | Extracted numeric claims vs. output from FRED API |
|Key Fact Coverage | Whether annotated key data points appear in the answer (data-extraction questions only) |

Runs Summary Evaluations with 3 metrics across all 9 agent variants and saves results.

| # | Model | Components |
|---|-------|-----------|
| 1 | gpt-4o-mini | full guide |
| 2 | gpt-4o-mini | semantic |
| 3 | gpt-4o-mini (fine-tuned)| semantic + checks |
| 4 | llama3.2 | full guide |
| 5 | llama3.2 | semantic |
| 6 | llama3.2 | semantic + date parser + checks |
| 7 | llama3.2 (fine-tuned)| full guide |
| 8 | llama3.2 (fine-tuned)| semantic |
| 9 | llama3.2 (fine-tuned)| semantic + date parser + checks |

Prompt Engineered agents:

| # | Model | Components |
|---|-------|-----------|
| 1 | llama3.2 (few-shot)| full guide |
| 2 | llama3.2 (few-shot)| semantic |
| 3 | llama3.2 (few-shot)| semantic + date parser + checks |

In [ ]:
from summary_evaluation import evaluate_summary_factuality
import json
from bert_score import score
import os
import pandas as pd

base_dir = os.getcwd()

TEST_FILE   = os.path.join(base_dir, "../data/QA_test.json")
LLAMA_DIR   = os.path.join(base_dir, "../files/llama3.2")
GPT_DIR     = os.path.join(base_dir, "../files/gpt-4o-mini")
REFERENCE_FILE = os.path.join(base_dir, "../files/human_generated_summary_test.json")

In [2]:
import numpy as np

with open(REFERENCE_FILE, encoding="utf-8") as f:
    human_file = json.load(f)
ref_lengths = [len(r.get("answer", "")) for r in human_file]
print(f'Avg length of reference summaries: {np.mean(ref_lengths)}.')

Avg length of reference summaries: 483.24.


In [3]:
os.makedirs(GPT_DIR + "/summary_eval", exist_ok=True)
os.makedirs(LLAMA_DIR + "/summary_eval", exist_ok=True)

In [4]:
def run_all_tests(
    api_file_path,
    out_path,
    test_file_path=TEST_FILE,
    human_ref_path=REFERENCE_FILE,
    compute_bertscore=True,
):
    """
    Run all summary evaluations on a single agent result file.

    Args:
        api_file_path:     Path to the agent result JSON (list of result dicts).
        out_path:          Path to save the evaluation result JSON.
        test_file_path:    Path to QA_test.json.
        human_ref_path:      Path to human-written reference answers (used for BERTScore).
        compute_bertscore: Whether to compute BERTScore (LLaMA only, default True).

    Returns:
        dict with avg_key_fact_coverage, avg_hallucination_rate,
        and avg_bertscore (if compute_bertscore=True).
    """
    with open(test_file_path, encoding="utf-8") as f:
        test_cases = json.load(f)
    with open(api_file_path, encoding="utf-8") as f:
        api_results_all = json.load(f)

    # load human reference only when needed
    human_answers = []
    if compute_bertscore:
        with open(human_ref_path, encoding="utf-8") as f:
            human_file = json.load(f)
        human_answers = [r.get("answer", "") for r in human_file]

    # print(f"\n{'#'*60}")
    # print(f"Running {len(test_cases)} test cases...")
    # print(f"{'#'*60}")

    per_question_results = []
    cand_answers = []   # for BERTScore, aligned with human_answers
    ref_answers  = []
    lengths = []

    for i, test_case in enumerate(test_cases):
        # print(f"\n[Test {i+1}/{len(test_cases)}] {test_case.get('question_id', '')}")

        result_entry = api_results_all[i] if i < len(api_results_all) else {}

        answer = result_entry.get("final_answer", "")
        lengths.append(len(answer))
        api_results = result_entry.get("api_results", [])
        key_facts  = test_case.get("key_facts", None)

        factuality = evaluate_summary_factuality(
            answer=answer,
            api_results=api_results,
            key_facts=key_facts,
            compact=True
        )
        kf = factuality["key_fact_coverage"]
        nf = factuality["numeric_faithfulness"]

        per_question_results.append({
            "question_id":        test_case.get("question_id", i),
            "question":           test_case.get("question", ""),
            "key_fact_coverage":  kf,
            "hallucination_rate": nf,
        })

        # collect for BERTScore — skip failed/empty answers
        if compute_bertscore and answer:
            cand_answers.append(answer)
            ref_answers.append(human_answers[i] if i < len(human_answers) else "")

    kf_scores = [r["key_fact_coverage"]  for r in per_question_results]
    nf_scores = [r["hallucination_rate"] for r in per_question_results]
    avg_kf = sum(kf_scores) / len(kf_scores) if kf_scores else 0.0
    avg_nf = sum(nf_scores) / len(nf_scores) if nf_scores else 0.0
    avg_len = np.mean(lengths)

    print(f"\nKey Fact Coverage Rate : {avg_kf:.4f}")
    print(f"Hallucination Rate     : {avg_nf:.4f}")

    summary = {
        "avg_key_fact_coverage":  round(avg_kf, 4),
        "avg_hallucination_rate": round(avg_nf, 4),
        "avg_length": round(avg_len, 4)
    }

    if compute_bertscore and cand_answers:
        # only score pairs where both sides are non-empty
        MAX_CHARS = 512 * 4  # ~512 tokens at ~4 chars/token
        valid_pairs = [
            (c[:MAX_CHARS], r[:MAX_CHARS])
            for c, r in zip(cand_answers, ref_answers)
            if c and r
        ]
        valid_ids = [
            tc.get("question_id", i)
            for i, (tc, c, r) in enumerate(zip(test_cases, cand_answers, ref_answers))
            if c and r
        ]
        if valid_pairs:
            cands, refs = zip(*valid_pairs)
            _, _, F1 = score(
                cands=list(cands),
                refs=list(refs),
                lang="en",
                model_type="roberta-large",  # 512 token limit, stable
                batch_size=8,
                verbose=False
            )
            avg_bs = round(F1.mean().item(), 4)
            bs_list = F1.tolist()
        else:
            avg_bs = 0.0
            bs_list = []

        print(f"BERTScore (F1)         : {avg_bs:.4f}")
        summary["avg_bertscore"] = avg_bs

        # attach per-question BERTScore
        id_to_bs = dict(zip(valid_ids, bs_list))
        for i, r in enumerate(per_question_results):
            qid = r.get("question_id", i)
            r["bertscore"] = round(id_to_bs[qid], 4) if qid in id_to_bs else None

    # save results
    output = {"summary": summary, "results": per_question_results}
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"\nResults saved to {out_path}")

    return summary

---
## 1. GPT-4o-mini — Baseline (full guide)

In [5]:
res_gpt_base = run_all_tests(
    api_file_path=f"{GPT_DIR}/QA_test_gpt_api.json",
    out_path=f"{GPT_DIR}/summary_eval/gpt_api.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.9304
Hallucination Rate     : 0.2467


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8632

Results saved to files/gpt-4o-mini/summary_eval/gpt_api.json


---
## 2. GPT-4o-mini — Semantic Retriever

In [6]:
res_gpt_sem = run_all_tests(
    api_file_path=f"{GPT_DIR}/QA_test_gpt_api_semantic_retriever.json",
    out_path=f"{GPT_DIR}/summary_eval/gpt_api_semantic_retriever.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.9667
Hallucination Rate     : 0.2225


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8656

Results saved to files/gpt-4o-mini/summary_eval/gpt_api_semantic_retriever.json


---
## 3. GPT-4o-mini — Final (semantic + date parser + checks)

In [7]:
res_gpt_final = run_all_tests(
    api_file_path=f"{GPT_DIR}/QA_test_gpt_api_final.json",
    out_path=f"{GPT_DIR}/summary_eval/gpt_api_final.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.9667
Hallucination Rate     : 0.2090


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8784

Results saved to files/gpt-4o-mini/summary_eval/gpt_api_final.json


---
## 4. LLaMA 3.2 — Baseline (full guide)

In [8]:
res_llama_base = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8750
Hallucination Rate     : 0.4346


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8566

Results saved to files/llama3.2/summary_eval/llama_api.json


---
## 5. LLaMA 3.2 — Semantic Retriever

In [9]:
res_llama_sem = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8650
Hallucination Rate     : 0.2516


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8620

Results saved to files/llama3.2/summary_eval/llama_api_semantic_retriever.json


---
## 6. LLaMA 3.2 — Final (semantic + date parser + checks)

In [10]:
res_llama_final = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8781
Hallucination Rate     : 0.3074


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8609

Results saved to files/llama3.2/summary_eval/llama_api_final.json


---
## 7. LLaMA 3.2 — Fine-tuned variants

Update the file paths below to point to your fine-tuned model result files.

In [11]:
# tool + summary
res_llama_ft_base3 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_finetuned3.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_finetuned3.json",
    compute_bertscore=True
)

res_llama_ft_sem3 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever_finetuned3.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever_finetuned3.json",
    compute_bertscore=True
)

res_llama_ft_final3 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final_finetuned3.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final_finetuned3.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8110
Hallucination Rate     : 0.4307


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8386

Results saved to files/llama3.2/summary_eval/llama_api_finetuned3.json

Key Fact Coverage Rate : 0.8283
Hallucination Rate     : 0.4902


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8449

Results saved to files/llama3.2/summary_eval/llama_api_semantic_retriever_finetuned3.json

Key Fact Coverage Rate : 0.8614
Hallucination Rate     : 0.4231


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8569

Results saved to files/llama3.2/summary_eval/llama_api_final_finetuned3.json


In [12]:
# summary only
res_llama_ft_base4 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_finetuned4.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_finetuned4.json",
    compute_bertscore=True
)

res_llama_ft_sem4 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever_finetuned4.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever_finetuned4.json",
    compute_bertscore=True
)

res_llama_ft_final4 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final_finetuned4.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final_finetuned4.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8162
Hallucination Rate     : 0.6569


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8420

Results saved to files/llama3.2/summary_eval/llama_api_finetuned4.json

Key Fact Coverage Rate : 0.8308
Hallucination Rate     : 0.5639


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8269

Results saved to files/llama3.2/summary_eval/llama_api_semantic_retriever_finetuned4.json

Key Fact Coverage Rate : 0.7917
Hallucination Rate     : 0.6208


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8184

Results saved to files/llama3.2/summary_eval/llama_api_final_finetuned4.json


In [13]:
# tool + summary
res_llama_ft_base5 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_finetuned5.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_finetuned5.json",
    compute_bertscore=True
)

res_llama_ft_sem5 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever_finetuned5.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever_finetuned5.json",
    compute_bertscore=True
)

res_llama_ft_final5 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final_finetuned5.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final_finetuned5.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8100
Hallucination Rate     : 0.3898


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8447

Results saved to files/llama3.2/summary_eval/llama_api_finetuned5.json

Key Fact Coverage Rate : 0.8824
Hallucination Rate     : 0.3553


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8529

Results saved to files/llama3.2/summary_eval/llama_api_semantic_retriever_finetuned5.json

Key Fact Coverage Rate : 0.8923
Hallucination Rate     : 0.3074


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8568

Results saved to files/llama3.2/summary_eval/llama_api_final_finetuned5.json


In [28]:
# tool + summary
res_llama_ft_base6 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_finetuned6.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_finetuned6.json",
    compute_bertscore=True
)

res_llama_ft_sem6 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever_finetuned6.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever_finetuned6.json",
    compute_bertscore=True
)

res_llama_ft_final6 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final_finetuned6.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final_finetuned6.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8667
Hallucination Rate     : 0.4305


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8341

Results saved to files/llama3.2/summary_eval/llama_api_finetuned6.json

Key Fact Coverage Rate : 0.8462
Hallucination Rate     : 0.2924


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8457

Results saved to files/llama3.2/summary_eval/llama_api_semantic_retriever_finetuned6.json

Key Fact Coverage Rate : 0.8681
Hallucination Rate     : 0.3691


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8421

Results saved to files/llama3.2/summary_eval/llama_api_final_finetuned6.json


## 8. Few-shot Prompt

In [15]:
res_llama_fs_base = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_few_shot.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_few_shot.json",
    compute_bertscore=True
)

res_llama_fs_sem = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever_few_shot.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever_few_shot.json",
    compute_bertscore=True
)

res_llama_fs_final = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final_few_shot.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final_few_shot.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8377
Hallucination Rate     : 0.4172


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8557

Results saved to files/llama3.2/summary_eval/llama_api_few_shot.json

Key Fact Coverage Rate : 0.8856
Hallucination Rate     : 0.3156


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8626

Results saved to files/llama3.2/summary_eval/llama_api_semantic_retriever_few_shot.json

Key Fact Coverage Rate : 0.9088
Hallucination Rate     : 0.2598


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8674

Results saved to files/llama3.2/summary_eval/llama_api_final_few_shot.json


---
## 9. Summary Comparison

In [16]:
rows = [
    ("gpt_api",                        res_gpt_base),
    ("gpt_api_semantic_retriever",      res_gpt_sem),
    ("gpt_api_final",                   res_gpt_final),
    ("llama_api",                       res_llama_base),
    ("llama_api_semantic_retriever",    res_llama_sem),
    ("llama_api_final",                 res_llama_final),
    # ("llama_api_finetuned3",             res_llama_ft_base3),
    # ("llama_api_semantic_finetuned3",    res_llama_ft_sem3),
    # ("llama_api_final_finetuned3",       res_llama_ft_final3),
    # ("llama_api_finetuned4",             res_llama_ft_base4),
    # ("llama_api_semantic_finetuned4",    res_llama_ft_sem4),
    # ("llama_api_final_finetuned4",       res_llama_ft_final4),
    ("llama_api_finetuned5",             res_llama_ft_base5),
    ("llama_api_semantic_finetuned5",    res_llama_ft_sem5),
    ("llama_api_final_finetuned5",       res_llama_ft_final5),
]

records = []
for name, res in rows:
    records.append({
        "agent":                name,
        "key_fact_coverage":   res.get("avg_key_fact_coverage",  "-"),
        "hallucination_rate":  res.get("avg_hallucination_rate", "-"),
        "average_length":      res.get("avg_length", "-"),
        "bertscore_f1":        res.get("avg_bertscore",          "N/A"),
    })

df = pd.DataFrame(records).set_index("agent")
df

,key_fact_coverage,hallucination_rate,average_length,bertscore_f1
agent,,,,
gpt_api,0.9304,0.2467,1258.96,0.8632
gpt_api_semantic_retriever,0.9667,0.2225,1231.70,0.8656
gpt_api_final,0.9667,0.2090,1246.18,0.8784
llama_api,0.8750,0.4346,719.16,0.8566
llama_api_semantic_retriever,0.8650,0.2516,710.68,0.8620
llama_api_final,0.8781,0.3074,883.48,0.8609
llama_api_finetuned5,0.8100,0.3898,734.68,0.8447
llama_api_semantic_finetuned5,0.8824,0.3553,869.42,0.8529
llama_api_final_finetuned5,0.8923,0.3074,863.26,0.8568


In [17]:
rows = [
    ("llama_api_few_shot",             res_llama_fs_base),
    ("llama_api_semantic_few_shot",    res_llama_fs_sem),
    ("llama_api_final_few_shot",       res_llama_fs_final),
]

records = []
for name, res in rows:
    records.append({
        "agent":                name,
        "key_fact_coverage":   res.get("avg_key_fact_coverage",  "-"),
        "hallucination_rate":  res.get("avg_hallucination_rate", "-"),
        "average_length":      res.get("avg_length", "-"),
        "bertscore_f1":        res.get("avg_bertscore",          "N/A"),
    })

df = pd.DataFrame(records).set_index("agent")
df

,key_fact_coverage,hallucination_rate,average_length,bertscore_f1
agent,,,,
llama_api_few_shot,0.8377,0.4172,695.32,0.8557
llama_api_semantic_few_shot,0.8856,0.3156,744.20,0.8626
llama_api_final_few_shot,0.9088,0.2598,600.28,0.8674


In [29]:
rows = [
    ("llama_api_finetuned3",             res_llama_ft_base3),
    ("llama_api_semantic_finetuned3",    res_llama_ft_sem3),
    ("llama_api_final_finetuned3",       res_llama_ft_final3),
    ("llama_api_finetuned4",             res_llama_ft_base4),
    ("llama_api_semantic_finetuned4",    res_llama_ft_sem4),
    ("llama_api_final_finetuned4",       res_llama_ft_final4),
    ("llama_api_finetuned5",             res_llama_ft_base5),
    ("llama_api_semantic_finetuned5",    res_llama_ft_sem5),
    ("llama_api_final_finetuned5",       res_llama_ft_final5),
    ("llama_api_finetuned6",             res_llama_ft_base6),
    ("llama_api_semantic_finetuned6",    res_llama_ft_sem6),
    ("llama_api_final_finetuned6",       res_llama_ft_final6),
]

records = []
for name, res in rows:
    records.append({
        "agent":                name,
        "key_fact_coverage":   res.get("avg_key_fact_coverage",  "-"),
        "hallucination_rate":  res.get("avg_hallucination_rate", "-"),
        "average_length":      res.get("avg_length", "-"),
        "bertscore_f1":        res.get("avg_bertscore",          "N/A"),
    })

df = pd.DataFrame(records).set_index("agent")
df

,key_fact_coverage,hallucination_rate,average_length,bertscore_f1
agent,,,,
llama_api_finetuned3,0.8110,0.4307,578.42,0.8386
llama_api_semantic_finetuned3,0.8283,0.4902,639.12,0.8449
llama_api_final_finetuned3,0.8614,0.4231,548.76,0.8569
llama_api_finetuned4,0.8162,0.6569,830.52,0.8420
llama_api_semantic_finetuned4,0.8308,0.5639,769.50,0.8269
llama_api_final_finetuned4,0.7917,0.6208,585.42,0.8184
llama_api_finetuned5,0.8100,0.3898,734.68,0.8447
llama_api_semantic_finetuned5,0.8824,0.3553,869.42,0.8529
llama_api_final_finetuned5,0.8923,0.3074,863.26,0.8568


## 10. Ablation Experiments

In [31]:
res_llama_checks_only = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_checks_only.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_checks_only.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8667
Hallucination Rate     : 0.3223


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8738

Results saved to files/llama3.2/summary_eval/llama_api_checks_only.json


In [32]:
res_llama_date_parser_only = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_date_only.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_date_only.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8547
Hallucination Rate     : 0.4099


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8744

Results saved to files/llama3.2/summary_eval/llama_api_date_only.json


In [34]:
rows = [
    ("llama_api_checks_only",             res_llama_checks_only),
    ("llama_api_date_parser_only",    res_llama_date_parser_only),
]

records = []
for name, res in rows:
    records.append({
        "agent":                name,
        "key_fact_coverage":   res.get("avg_key_fact_coverage",  "-"),
        "hallucination_rate":  res.get("avg_hallucination_rate", "-"),
        "average_length":      res.get("avg_length", "-"),
        "bertscore_f1":        res.get("avg_bertscore",          "N/A"),
    })

df = pd.DataFrame(records).set_index("agent")
df

,key_fact_coverage,hallucination_rate,average_length,bertscore_f1
agent,,,,
llama_api_checks_only,0.8667,0.3223,571.46,0.8738
llama_api_date_parser_only,0.8547,0.4099,566.70,0.8744
